# S-DoT UTCI → 네트워크 링크 IDW 보간

**목적**: 성동구 57개 S-DoT 센서의 시간대별 UTCI를 도로 네트워크 링크(엣지) 단위로 공간 보간한다.

## 입력
- `../04_분석결과/sdot_utci_seongdong.csv` — 센서별 시간대별 UTCI
- `../01_네트워크/seongdong_walk_network.graphml` — 도로 네트워크

## 출력
- `../04_분석결과/link_utci_by_hour.csv` — 링크별 × 시간대별 UTCI (7일 평균)

## IDW 보간 방식
```
UTCI_link(h) = Σ(UTCI_s(h) / d²_s) / Σ(1 / d²_s)

- d_s : 링크 중점 ↔ 센서 s 간 거리 (EPSG:5186 기준, 미터)
- 멱지수 p = 2 (표준 IDW)
- 센서가 링크 중점과 겹칠 경우(d < 1m): 해당 센서 UTCI 직접 사용
```

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from pathlib import Path
from pyproj import Transformer

NETWORK_FILE = Path('../01_네트워크/seongdong_walk_network.graphml')
UTCI_FILE    = Path('../04_분석결과/sdot_utci_seongdong.csv')
OUT_DIR      = Path('../04_분석결과')

IDW_POWER = 2

# 좌표 변환기: WGS84 → EPSG:5186
wgs_to_5186 = Transformer.from_crs('EPSG:4326', 'EPSG:5186', always_xy=True)

print('설정 완료')

## 1. 네트워크 로드 → 링크 중점 좌표 추출

In [ ]:
G = nx.read_graphml(NETWORK_FILE)
print(f'노드: {G.number_of_nodes():,} / 엣지: {G.number_of_edges():,}')

# 노드 좌표 딕셔너리
node_xy = {}
for nid, d in G.nodes(data=True):
    x5186, y5186 = wgs_to_5186.transform(float(d['x']), float(d['y']))
    node_xy[nid] = (x5186, y5186)

# 엣지별 중점 좌표 + 속성
edge_rows = []
for u, v, d in G.edges(data=True):
    x_u, y_u = node_xy[u]
    x_v, y_v = node_xy[v]
    mx, my = (x_u + x_v) / 2, (y_u + y_v) / 2  # 링크 중점
    edge_rows.append({
        'u': u, 'v': v,
        'mx': mx, 'my': my,
        'highway': d.get('highway', 'unknown'),
        'bridge': d.get('bridge', 'no'),
        'length': float(d.get('length', 0)),
        'name': d.get('name', '')
    })

edges_df = pd.DataFrame(edge_rows)
print(f'링크 수: {len(edges_df):,}')
print(f'교량 링크: {(edges_df["bridge"] == "yes").sum()}개')
edges_df.head(3)

## 2. S-DoT UTCI 로드 → 시간대별 센서 평균 (7일 평균)

In [ ]:
utci_df = pd.read_csv(UTCI_FILE, encoding='utf-8-sig')

# 센서별 × 시간대별 평균 UTCI (7일 평균)
sensor_hourly = (
    utci_df
    .groupby(['serial', 'hour', 'lat', 'lon'])['utci_val']
    .mean()
    .reset_index()
    .rename(columns={'utci_val': 'utci_mean'})
)

# 센서 좌표 → EPSG:5186
sx, sy = wgs_to_5186.transform(
    sensor_hourly['lon'].values,
    sensor_hourly['lat'].values
)
sensor_hourly['sx'] = sx
sensor_hourly['sy'] = sy

print(f'센서-시간대 조합: {len(sensor_hourly):,}개')
print(f'시간대 수: {sensor_hourly["hour"].nunique()} / 센서 수: {sensor_hourly["serial"].nunique()}')
sensor_hourly.head(3)

## 3. IDW 보간 함수

In [ ]:
def idw_interpolate(qx, qy, sensor_x, sensor_y, sensor_vals, power=2):
    """쿼리점 (qx, qy)에서 IDW로 UTCI 추정."""
    dx = sensor_x - qx
    dy = sensor_y - qy
    dist = np.sqrt(dx**2 + dy**2)

    # 거의 같은 위치 센서 있으면 직접 반환
    if dist.min() < 1.0:
        return sensor_vals[dist.argmin()]

    weights = 1.0 / (dist ** power)
    return np.sum(weights * sensor_vals) / np.sum(weights)

print('IDW 함수 정의 완료')

## 4. 링크 × 시간대 IDW 보간 실행

In [ ]:
hours = sorted(sensor_hourly['hour'].unique())
link_utci_rows = []

edge_mx = edges_df['mx'].values
edge_my = edges_df['my'].values

for hour in hours:
    s = sensor_hourly[sensor_hourly['hour'] == hour]
    sx_arr = s['sx'].values
    sy_arr = s['sy'].values
    uv_arr = s['utci_mean'].values

    # 유효 센서만 (결측 제거)
    valid = ~np.isnan(uv_arr)
    if valid.sum() == 0:
        continue
    sx_v, sy_v, uv_v = sx_arr[valid], sy_arr[valid], uv_arr[valid]

    # 전체 링크에 한 번에 벡터화 계산
    utci_links = np.array([
        idw_interpolate(mx, my, sx_v, sy_v, uv_v, IDW_POWER)
        for mx, my in zip(edge_mx, edge_my)
    ])

    df_h = edges_df[['u', 'v', 'highway', 'bridge', 'length', 'name']].copy()
    df_h['hour'] = hour
    df_h['utci_idw'] = utci_links.round(2)
    link_utci_rows.append(df_h)

    print(f'{hour:02d}시 완료 — 사용 센서 {valid.sum()}개, 링크 UTCI 평균 {utci_links.mean():.1f}°C')

link_utci = pd.concat(link_utci_rows, ignore_index=True)
print(f'\n총 {len(link_utci):,}행 (링크 {len(edges_df):,} × 시간대 {len(hours)})')

## 5. UTCI 열 스트레스 등급 + 속도 감소 계수 부여

In [ ]:
def utci_speed_factor(row):
    """
    UTCI 기반 보행 속도 감소 계수.
    교량(완전 노출)은 추가 패널티 적용.
    """
    u = row['utci_idw']
    is_bridge = (row['bridge'] == 'yes')

    # 기본 계수
    if u < 26:   factor = 1.00   # 쾌적
    elif u < 32: factor = 0.90   # 약한 열
    elif u < 38: factor = 0.75   # 강한 열
    elif u < 46: factor = 0.50   # 매우 강한 열
    else:        factor = 0.10   # 극한 → 사실상 통행 불가

    # 교량 추가 패널티 (그늘 없음, 복사 노출)
    if is_bridge:
        factor = max(factor * 0.7, 0.05)

    return round(factor, 3)

def utci_category(u):
    if u < 9:    return 'cold'
    if u < 26:   return 'comfortable'
    if u < 32:   return 'moderate_heat'
    if u < 38:   return 'strong_heat'
    if u < 46:   return 'very_strong_heat'
    return 'extreme_heat'

link_utci['utci_cat']      = link_utci['utci_idw'].apply(utci_category)
link_utci['speed_factor']  = link_utci.apply(utci_speed_factor, axis=1)

# 시간대별 교량 링크 통계 확인
bridge_stats = (
    link_utci[link_utci['bridge'] == 'yes']
    .groupby('hour')[['utci_idw', 'speed_factor']]
    .mean().round(2)
)
print('교량 링크 시간대별 UTCI / 속도계수:')
print(bridge_stats.to_string())

## 6. 시각화 — 핵심 3개 시간대 (07 / 13 / 20시)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 링크를 GeoDataFrame으로 변환 (시각화용)
from shapely.geometry import LineString

def make_link_gdf(hour_val):
    sub = link_utci[link_utci['hour'] == hour_val].copy()
    geoms = []
    for _, row in sub.iterrows():
        xu, yu = node_xy[row['u']]
        xv, yv = node_xy[row['v']]
        geoms.append(LineString([(xu, yu), (xv, yv)]))
    sub['geometry'] = geoms
    return gpd.GeoDataFrame(sub, geometry='geometry', crs='EPSG:5186')

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
titles = ['07시 (이른 아침)', '13시 (폭염 피크)', '20시 (저녁)']
plot_hours = [7, 13, 20]

for ax, h, title in zip(axes, plot_hours, titles):
    gdf_h = make_link_gdf(h)
    gdf_h.plot(
        column='utci_idw', ax=ax,
        cmap='RdYlGn_r', linewidth=0.6,
        vmin=28, vmax=42,
        legend=(h == 13),
        legend_kwds={'label': 'UTCI (°C)', 'shrink': 0.6}
    )
    # 교량 강조
    gdf_h[gdf_h['bridge'] == 'yes'].plot(
        ax=ax, color='blue', linewidth=2.0, label='교량'
    )
    avg_utci = gdf_h['utci_idw'].mean()
    ax.set_title(f'{title}\n평균 UTCI {avg_utci:.1f}°C', fontsize=11)
    ax.set_axis_off()

plt.suptitle('성동구 도로 링크별 UTCI (2025.07.28~08.03 평균, 파란선=교량)', fontsize=13)
plt.tight_layout()
plt.savefig('../05_시각화/link_utci_3hours.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 05_시각화/link_utci_3hours.png')

## 7. 결과 저장

In [ ]:
out_path = OUT_DIR / 'link_utci_by_hour.csv'
link_utci.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'저장: {out_path}')
print(f'  행 수: {len(link_utci):,}')
print(f'  컬럼: {link_utci.columns.tolist()}')

# 핵심 시간대 요약
print('\n핵심 시간대 링크 평균 UTCI:')
for h in [7, 10, 13, 17, 20]:
    sub = link_utci[link_utci['hour'] == h]
    all_avg  = sub['utci_idw'].mean()
    brg_avg  = sub[sub['bridge']=='yes']['utci_idw'].mean()
    brg_sf   = sub[sub['bridge']=='yes']['speed_factor'].mean()
    print(f'  {h:02d}시 — 전체 {all_avg:.1f}°C | 교량 {brg_avg:.1f}°C (속도계수 {brg_sf:.2f})')